# Notebook 4: Results Analysis

Comprehensive results analysis:
- Full comparison table (all metrics)
- Throughput gain breakdown
- Per-SNR performance
- MCS error analysis (conservative vs. aggressive)
- Cross-channel generalisation
- Statistical significance testing

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

from src.channel.simulator import (
    ChannelSimulator, ChannelConfig,
    bler_model, effective_throughput, NUM_MCS
)
from src.models.supervised import CQIBaseline, RFLinkAdapter, XGBLinkAdapter, LSTMWrapper
from src.models.dqn_agent import DQNAgent, LinkAdaptationEnv
from src.evaluation.metrics import LinkAdaptationEvaluator
from src.evaluation.visualize import (
    plot_throughput_comparison, plot_bler_vs_snr,
    plot_per_snr_throughput, plot_mcs_distribution,
    plot_confusion_matrix
)

sns.set_theme(style='whitegrid', font_scale=1.1)
print('Imports OK')

## 1. Load / Train Models and Run Full Evaluation

In [ ]:
CHANNEL = 'rayleigh'
cfg = ChannelConfig(channel_type=CHANNEL, snr_mean_db=15.0, snr_std_db=6.0, seed=42)
sim = ChannelSimulator(cfg)
ds  = sim.generate_dataset(n_frames=80_000, window=8)

idx_tr, idx_val, idx_te = ds.train_val_test_split()
X_train, y_train = ds.features[idx_tr],  ds.labels_mcs[idx_tr]
X_val,   y_val   = ds.features[idx_val], ds.labels_mcs[idx_val]
X_test,  y_test  = ds.features[idx_te],  ds.labels_mcs[idx_te]
snr_test = ds.snr_trace[idx_te + 8]

# Train models
models = {}
predictions = {}

cqi = CQIBaseline(); models['CQI Baseline'] = cqi
predictions['CQI Baseline'] = cqi.predict(X_test)

rf = RFLinkAdapter(n_estimators=300).fit(X_train, y_train)
models['Random Forest'] = rf
predictions['Random Forest'] = rf.predict(X_test)

xgb_m = XGBLinkAdapter(n_estimators=400).fit(X_train, y_train, X_val, y_val)
models['XGBoost'] = xgb_m
predictions['XGBoost'] = xgb_m.predict(X_test)

lstm = LSTMWrapper(epochs=20).fit(X_train, y_train, X_val, y_val, verbose=False)
models['LSTM'] = lstm
predictions['LSTM'] = lstm.predict(X_test)

env = LinkAdaptationEnv(ds.features[idx_tr], ds.snr_trace[idx_tr + 8], CHANNEL)
agent = DQNAgent(state_dim=ds.features.shape[1])
agent.train(env, n_episodes=15, verbose=False)
predictions['DQN'] = np.array([
    agent.select_action(X_test[i], greedy=True) for i in range(len(X_test))])

print('All models trained and evaluated.')

## 2. Full Metrics Table

In [ ]:
evaluator = LinkAdaptationEvaluator(snr_test, y_test, CHANNEL)
model_results = [evaluator.evaluate(preds, name)
                 for name, preds in predictions.items()]

comparison = evaluator.compare(model_results)
print('\n=== Full Results Table ===')
print(comparison.to_string())

oracle = evaluator.oracle_metrics()
print(f'\nOracle throughput: {oracle["mean_throughput_mbps"]:.2f} Mbps')

## 3. Throughput Gain Breakdown

In [ ]:
# Waterfall chart: how much does each model contribute?
model_names = [r.name for r in model_results]
tputs = [r.mean_throughput_mbps for r in model_results]
cqi_tput = model_results[0].mean_throughput_mbps

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Absolute throughput
colors = ['#937860', '#4C72B0', '#8172B2', '#55A868', '#C44E52']
axes[0].barh(model_names, tputs, color=colors, alpha=0.85, height=0.55)
axes[0].axvline(oracle['mean_throughput_mbps'], color='black',
                linestyle='--', linewidth=1.5, label='Oracle')
for i, (name, tput) in enumerate(zip(model_names, tputs)):
    axes[0].text(tput + 0.3, i, f'{tput:.1f}', va='center', fontsize=9)
axes[0].set_xlabel('Mean Throughput (Mbps)')
axes[0].set_title('Absolute Throughput Comparison')
axes[0].legend()
axes[0].grid(axis='x', alpha=0.3)

# Gain over CQI baseline
gains = [(t - cqi_tput) / cqi_tput * 100 for t in tputs]
gain_colors = ['#937860'] + ['#4C72B0' if g > 0 else '#C44E52' for g in gains[1:]]
axes[1].barh(model_names, gains, color=gain_colors, alpha=0.85, height=0.55)
axes[1].axvline(0, color='black', linewidth=1)
for i, g in enumerate(gains):
    axes[1].text(g + 0.2 if g >= 0 else g - 0.2, i,
                 f'{g:+.1f}%', va='center',
                 ha='left' if g >= 0 else 'right', fontsize=9)
axes[1].set_xlabel('Throughput Gain over CQI Baseline (%)')
axes[1].set_title('Relative Throughput Gain')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/plots/nb4_throughput_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. MCS Error Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MCS prediction error distribution
ax = axes[0]
for r, color in zip(model_results, colors):
    errors = r.predictions - r.true_labels
    ax.hist(errors, bins=range(-10, 11), alpha=0.6, label=r.name,
            color=color, density=True, histtype='step', linewidth=2)
ax.axvline(0, color='black', linewidth=1.5, label='Perfect')
ax.set_xlabel('MCS Prediction Error (pred − optimal)')
ax.set_ylabel('Density')
ax.set_title('Distribution of MCS Prediction Error')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Conservative vs Aggressive bias
ax = axes[1]
conservative = [r.conservative_bias for r in model_results]
aggressive   = [r.aggressive_bias   for r in model_results]
exact        = [100 - c - a for c, a in zip(conservative, aggressive)]

x = np.arange(len(model_names))
w = 0.25
ax.bar(x - w, conservative, w, label='Conservative (under-MCS)', color='#4C72B0', alpha=0.85)
ax.bar(x,     exact,        w, label='Exact',                    color='#55A868', alpha=0.85)
ax.bar(x + w, aggressive,   w, label='Aggressive (over-MCS)',    color='#C44E52', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=12, ha='right')
ax.set_ylabel('% of Predictions')
ax.set_title('Conservative vs. Aggressive MCS Bias')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/plots/nb4_mcs_error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Cross-Channel Generalisation Test

In [ ]:
# Test models trained on Rayleigh against AWGN and Jakes
cross_channel_results = {'Model': model_names}

for test_channel in ['awgn', 'rayleigh', 'jakes']:
    cfg_test = ChannelConfig(channel_type=test_channel, snr_mean_db=15.0, snr_std_db=6.0, seed=99)
    sim_test = ChannelSimulator(cfg_test)
    ds_test  = sim_test.generate_dataset(n_frames=20_000, window=8)
    _, _, idx_te_c = ds_test.train_val_test_split()
    X_te_c = ds_test.features[idx_te_c]
    y_te_c = ds_test.labels_mcs[idx_te_c]
    snr_te_c = ds_test.snr_trace[idx_te_c + 8]

    accs = []
    for name, m in models.items():
        preds = m.predict(X_te_c)
        from sklearn.metrics import accuracy_score
        accs.append(accuracy_score(y_te_c, preds) * 100)
    cross_channel_results[f'Acc {test_channel} (%)'] = accs

cross_df = pd.DataFrame(cross_channel_results).set_index('Model')
print('Cross-Channel Accuracy (models trained on Rayleigh):')
print(cross_df.to_string())

fig, ax = plt.subplots(figsize=(11, 5))
cross_df.plot(kind='bar', ax=ax, alpha=0.85, edgecolor='white',
              color=['#4C72B0', '#55A868', '#C44E52'])
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Cross-Channel Generalisation (trained on Rayleigh)')
ax.legend(title='Test channel')
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha='right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/plots/nb4_cross_channel.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary and Key Findings

In [ ]:
print('=' * 60)
print('SUMMARY: ML-Based Link Adaptation Results')
print('=' * 60)
print(f'Channel type : {CHANNEL}')
print(f'Test frames  : {len(X_test):,}')
print()

best = max(model_results, key=lambda r: r.mean_throughput_mbps)
cqi_r = model_results[0]

print(f'Best model       : {best.name}')
print(f'Best throughput  : {best.mean_throughput_mbps:.2f} Mbps')
print(f'CQI baseline     : {cqi_r.mean_throughput_mbps:.2f} Mbps')
print(f'Throughput gain  : +{(best.mean_throughput_mbps - cqi_r.mean_throughput_mbps) / cqi_r.mean_throughput_mbps * 100:.1f}%')
print(f'Oracle           : {oracle["mean_throughput_mbps"]:.2f} Mbps')
print(f'Gap to oracle    : {(oracle["mean_throughput_mbps"] - best.mean_throughput_mbps) / oracle["mean_throughput_mbps"] * 100:.1f}%')
print()
print(f'Best accuracy    : {best.accuracy*100:.2f}% ({best.name})')
print(f'CQI accuracy     : {cqi_r.accuracy*100:.2f}%')
print()
print('Key insight: The first temporal difference (snr_diff1) is the')
print('3rd most important feature — channel dynamics, not just level,')
print('are critical for accurate MCS selection.')